# Phase C2 — targeted acoustic review
Select one of the 11 review clips below. The diagnostic figure and all numerical metrics use the **original WAV**. The normalized copy is an amplitude-only **listening aid**; never use it for RMS or amplitude comparisons. Candidate peaks are acoustic activity, **not germination sounds**. Record only neutral human observations. Blank labels are intentionally unreviewed. The microphone covers the whole box and cannot identify a seed source.

The context table lists separate recordings and actual gaps. Listening to adjacent clips individually does not imply continuous audio.


In [ ]:
from pathlib import Path
import sys
PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
if not (PROJECT / 'src' / 'phase_c2_audio_review.py').exists(): raise RuntimeError('Open inside the seed-germination repository')
sys.path.insert(0, str(PROJECT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Audio, Image, Markdown, clear_output, display
from src.audio_features import read_wav_float
from src.phase_c2_audio_review import ALLOWED_LABELS, ALLOWED_CONFIDENCE, save_manual_review
TABLES = PROJECT / 'results/tables/targeted_multimodal'
MANIFEST = TABLES / 'targeted_audio_review_manifest.csv'
METRICS = TABLES / 'targeted_audio_detailed_metrics.csv'
CONTEXT = TABLES / 'targeted_audio_context.csv'
assert METRICS.is_file() and CONTEXT.is_file(), 'Run python scripts/run_phase_c2.py first'
print('Python:', sys.executable, '| review clips:', len(pd.read_csv(METRICS)))


## Review one clip and save a neutral label
Use the original playback, normalized listening copy, waveform, RMS envelope, spectrogram, and adjacent recordings. The save button backs up both CSVs before writing your label, confidence and notes. Existing decisions remain when diagnostics are regenerated. If nothing is clearly audible, describe that in notes; do not infer absence of an acoustic signal.


In [ ]:
def current_tables():
    return (pd.read_csv(METRICS, keep_default_na=False), pd.read_csv(CONTEXT, keep_default_na=False))
metrics, context = current_tables()
choice = widgets.Dropdown(options=[(f'{r.review_id} | {r.wav_filename} | {r.visual_group_id}', r.review_id) for r in metrics.itertuples(index=False)], description='Clip:', layout=widgets.Layout(width='850px'))
label = widgets.Dropdown(options=('', *ALLOWED_LABELS), description='Label:', layout=widgets.Layout(width='520px'))
confidence = widgets.Dropdown(options=('', *ALLOWED_CONFIDENCE), description='Confidence:', layout=widgets.Layout(width='350px'))
notes = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='850px', height='75px'))
save = widgets.Button(description='Save human review', button_style='success')
display_area = widgets.Output()
status_area = widgets.Output()
def render_selected(*_):
    current, neighbors = current_tables()
    row = current.loc[current.review_id.eq(choice.value)].iloc[0]
    label.value = row.manual_audio_label if row.manual_audio_label in label.options else ''
    confidence.value = row.manual_audio_confidence if row.manual_audio_confidence in confidence.options else ''
    notes.value = str(row.notes)
    lag_text = row.lag_seconds if str(row.lag_seconds) else 'not available'
    with display_area:
        clear_output(wait=True)
        display(Markdown(f'### {row.review_id} — {row.wav_filename}\n**{row.visual_group_id}**: {row.seed_ids} · {row.relationship_to_visual_event} · candidate lag: {lag_text} s'))
        display(current.loc[current.review_id.eq(choice.value), ['wav_timestamp','rms','max_abs_amplitude','mean_abs_amplitude','zero_crossing_rate_hz','spectral_centroid_hz','spectral_bandwidth_hz','dominant_frequency_hz','candidate_peak_count','strongest_peak_prominence','strongest_peak_timestamp','strongest_peak_width_seconds']])
        display(Image(filename=row.diagnostic_figure_path))
        display(Markdown('**Original WAV — measurements use this signal**'))
        display(Audio(filename=row.original_raw_path, normalize=False))
        display(Markdown('**Normalized listening copy — amplitude changed, listening only**'))
        display(Audio(filename=row.listening_copy_path, normalize=False))
        local = neighbors.loc[neighbors.review_id.eq(choice.value), ['role','wav_filename','start_timestamp','end_timestamp','start_lag_from_target_seconds','unrecorded_gap_from_previous_seconds','original_raw_path']]
        display(Markdown('**Separate adjacent recordings and actual gaps (seconds)**'))
        display(local.drop(columns='original_raw_path'))
        for neighbor in local.itertuples(index=False):
            if neighbor.role != 'target':
                display(Markdown(f'{neighbor.role}: {neighbor.wav_filename}'))
                display(Audio(filename=neighbor.original_raw_path, normalize=False))
        if row.manual_audio_label in ('uncertain', 'short_transient_unexplained'):
            rate, original = read_wav_float(Path(row.original_raw_path))
            _, normalized = read_wav_float(Path(row.listening_copy_path))
            fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
            t = np.arange(len(original))/rate
            axes[0].plot(t, original.mean(axis=1), lw=.5); axes[0].set_ylabel('Original amplitude')
            axes[1].plot(t, normalized.mean(axis=1), lw=.5); axes[1].set_ylabel('Listening copy')
            axes[1].set_xlabel('Seconds within WAV'); fig.tight_layout(); display(fig); plt.close(fig)
            controls = current.loc[current.relationship_to_visual_event.str.contains('control', case=False) & current.visual_group_id.eq(row.visual_group_id)]
            if len(controls):
                display(Markdown('**Low-activity control for comparison (original signal):**'))
                display(Image(filename=controls.iloc[0].diagnostic_figure_path))
def save_selected(_):
    with status_area:
        clear_output(wait=True)
        try:
            save_manual_review(MANIFEST, METRICS, choice.value, label.value, confidence.value, notes.value)
            print('Saved with timestamped CSV backups. Review summary updated.')
            render_selected()
        except Exception as error:
            print(type(error).__name__ + ': ' + str(error))
choice.observe(render_selected, names='value')
save.on_click(save_selected)
display(widgets.VBox([choice, label, confidence, notes, save, status_area, display_area]))
render_selected()
